# Brain Tumor MRI — Model Training

This notebook trains a **ResNet18** model (pretrained on ImageNet) to classify brain MRI scans into 4 categories:
- `glioma` — malignant glial cell tumor
- `meningioma` — tumor of the meninges
- `pituitary` — pituitary gland tumor
- `no_tumor` — healthy scan

**Hardware:** Google Colab T4/A100 GPU recommended  
**Framework:** PyTorch + torchvision  
**Epochs:** 10  |  **Batch size:** 32  |  **Optimizer:** Adam + StepLR

---

## Step 1: Google Colab Setup — Mount Drive & Check GPU

In [ ]:
# Mount Google Drive to persist the trained model
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/BrainTumorModel'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model will be saved to: {SAVE_DIR}")

In [ ]:
# Verify GPU availability
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if device.type == 'cuda':
    print(f"GPU Name  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA ver  : {torch.version.cuda}")
else:
    print("WARNING: No GPU found. Training will be slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU")

## Step 2: Install Dependencies & Set Paths

In [ ]:
!pip install kaggle --quiet

import os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT  = '/content/brain_tumor_dataset'
TRAIN_DIR  = os.path.join(DATA_ROOT, 'Training')
TEST_DIR   = os.path.join(DATA_ROOT, 'Testing')
MODEL_PATH = os.path.join(SAVE_DIR, 'resnet18_braintumor.pt')

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_CLASSES  = 4
BATCH_SIZE   = 32
NUM_EPOCHS   = 10
LEARNING_RATE = 1e-3
VAL_SPLIT    = 0.2
NUM_WORKERS  = 2

print("Configuration loaded.")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  Device      : {device}")

## Step 3: Data Transforms with Augmentation

**Training augmentation** helps the model generalize by exposing it to natural variations:
- Random horizontal flip
- Random rotation (±15°)
- Color jitter (brightness, contrast)
- Normalize with ImageNet mean/std (required for pretrained ResNet18)

**Validation/Test transforms** apply only resize + normalize (no stochastic augmentation).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Train transforms:")
print(train_transforms)
print("\nVal/Test transforms:")
print(val_transforms)

## Step 4: BrainTumorDataset — Load & Split

We use `torchvision.datasets.ImageFolder` and perform an 80/20 random split on the `Training/` directory. The `Testing/` directory is held out completely.

In [ ]:
from torch.utils.data import Subset
import copy

# ── Load full training set (with train transforms initially) ─────────────────
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
CLASS_NAMES = full_train_dataset.classes
print(f"Classes : {CLASS_NAMES}")
print(f"Total training samples: {len(full_train_dataset)}")

# ── Train / Val split ─────────────────────────────────────────────────────────
total   = len(full_train_dataset)
val_n   = int(VAL_SPLIT * total)
train_n = total - val_n

train_subset, val_subset = random_split(
    full_train_dataset, [train_n, val_n],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply val_transforms to the validation subset
# We do this by creating a wrapper dataset class
class TransformSubset(torch.utils.data.Dataset):
    """Wraps a Subset and applies a different transform."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
        # Expose classes from underlying ImageFolder
        self.classes        = subset.dataset.classes
        self.class_to_idx   = subset.dataset.class_to_idx

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Get raw PIL image by bypassing the dataset's transform
        orig_transform = self.subset.dataset.transform
        self.subset.dataset.transform = None
        img, label = self.subset[idx]
        self.subset.dataset.transform = orig_transform
        if self.transform:
            img = self.transform(img)
        return img, label

val_dataset  = TransformSubset(val_subset, val_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_transforms)

print(f"\nTrain subset  : {train_n} samples")
print(f"Val subset    : {val_n} samples")
print(f"Test set      : {len(test_dataset)} samples")

## Step 5: Create DataLoaders

In [ ]:
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# Verify a single batch shape
imgs, labels = next(iter(train_loader))
print(f"\nBatch image tensor shape : {imgs.shape}  (B, C, H, W)")
print(f"Batch label tensor shape : {labels.shape}")

## Step 6: Build the ResNet18 Model

We use **transfer learning**:
1. Load ResNet18 pretrained on ImageNet (1000-class weights)
2. Replace the final fully connected layer `fc` with a new `Linear(512, 4)`
3. All other layers are fine-tuned (not frozen)

The pretrained features from ImageNet generalize well to medical imaging.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Load pretrained ResNet18
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Replace the final layer
in_features = model.fc.in_features          # 512
model.fc = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# Print modified final layer
print(f"Original fc replaced with: {model.fc}")
print(f"\nTotal parameters     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 7: Loss Function, Optimizer & Scheduler

- **Loss:** CrossEntropyLoss (multi-class)
- **Optimizer:** Adam with lr=1e-3
- **Scheduler:** StepLR — decays LR by 0.1 every 4 epochs to fine-tune convergence

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

print(f"Loss      : CrossEntropyLoss")
print(f"Optimizer : Adam  (lr={LEARNING_RATE}, weight_decay=1e-4)")
print(f"Scheduler : StepLR  (step_size=4, gamma=0.1)")

# Expected LR schedule across epochs
print("\nLR schedule preview:")
lr_preview = LEARNING_RATE
for epoch in range(1, NUM_EPOCHS + 1):
    if epoch > 1 and (epoch - 1) % 4 == 0:
        lr_preview *= 0.1
    print(f"  Epoch {epoch:>2}: lr = {lr_preview:.2e}")

## Step 8: Training Loop

Full training loop with:
- Per-epoch train loss + validation accuracy tracking
- Best model checkpoint saving
- Time estimation per epoch

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Run one training epoch. Returns average loss."""
    model.train()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


def validate(model, loader, device):
    """Evaluate model on loader. Returns accuracy (0-100)."""
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds   = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# ── Training History ──────────────────────────────────────────────────────────
history = {
    'train_loss': [],
    'val_acc':    [],
    'lr':         [],
}

best_val_acc  = 0.0
best_model_wts = None

print(f"{'Epoch':>6} | {'Train Loss':>11} | {'Val Acc':>9} | {'LR':>10} | {'Time':>8}")
print("-" * 58)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_acc    = validate(model, val_loader, device)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Step the scheduler
    scheduler.step()

    elapsed = time.time() - t0

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        import copy
        best_model_wts = copy.deepcopy(model.state_dict())
        marker = '  ← best'
    else:
        marker = ''

    print(f"{epoch:>6} | {train_loss:>11.4f} | {val_acc:>8.2f}% | {current_lr:>10.2e} | {elapsed:>6.1f}s{marker}")

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

## Step 9: Save the Trained Model

Save the best checkpoint (highest validation accuracy) to Google Drive and a local copy.

In [ ]:
# Restore best weights
model.load_state_dict(best_model_wts)

# Save full checkpoint (weights + metadata)
checkpoint = {
    'model_state_dict': best_model_wts,
    'class_names':      CLASS_NAMES,
    'num_classes':      NUM_CLASSES,
    'best_val_acc':     best_val_acc,
    'history':          history,
    'hyperparams': {
        'batch_size':    BATCH_SIZE,
        'num_epochs':    NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'seed':          SEED,
    }
}

# Save to Google Drive
torch.save(checkpoint, MODEL_PATH)
print(f"Model saved to Google Drive: {MODEL_PATH}")

# Also save locally in /content for notebook 03
LOCAL_MODEL_PATH = '/content/resnet18_braintumor.pt'
torch.save(checkpoint, LOCAL_MODEL_PATH)
print(f"Local copy saved: {LOCAL_MODEL_PATH}")

# Verify file size
size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"Checkpoint size: {size_mb:.2f} MB")

## Step 10: Plot Training Curves

Visualize training loss and validation accuracy across all epochs.

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, NUM_EPOCHS + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ResNet18 — Brain Tumor MRI Training', fontsize=15, fontweight='bold')

# ── Left: Training Loss ───────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(epochs, history['train_loss'], 'o-', color='steelblue',
         linewidth=2, markersize=6, label='Train Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=13)
ax1.set_xticks(epochs)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Annotate minimum loss
min_loss_epoch = epochs[history['train_loss'].index(min(history['train_loss']))]
min_loss_val   = min(history['train_loss'])
ax1.annotate(f'Min: {min_loss_val:.4f}',
             xy=(min_loss_epoch, min_loss_val),
             xytext=(min_loss_epoch + 0.5, min_loss_val + 0.05),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=10, color='red')

# ── Right: Validation Accuracy ────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(epochs, history['val_acc'], 's-', color='coral',
         linewidth=2, markersize=6, label='Val Accuracy')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Validation Accuracy', fontsize=13)
ax2.set_xticks(epochs)
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

# Annotate best accuracy
best_epoch  = epochs[history['val_acc'].index(max(history['val_acc']))]
best_acc    = max(history['val_acc'])
ax2.axhline(y=best_acc, color='green', linestyle='--', alpha=0.5)
ax2.annotate(f'Best: {best_acc:.2f}%',
             xy=(best_epoch, best_acc),
             xytext=(best_epoch + 0.5, best_acc - 5),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=10, color='green')

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to /content/training_curves.png")

## Summary

| Item | Value |
|------|-------|
| Architecture | ResNet18 (pretrained ImageNet) |
| Final FC layer | Linear(512 → 4) |
| Epochs | 10 |
| Best Val Accuracy | see above |
| Checkpoint | `resnet18_braintumor.pt` |

**Next step:** Open `03_model_evaluation.ipynb` to evaluate the model on the test set.